# 14장. 장애·복구와 운영 관리

OpenAI-hosted 작업의 관찰 연결을 닫고 같은 Turn의 결과를 회수합니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | 관찰 재연결·저장 상태·입력 재전송 방지 |
| 이번 실습 조합 | OpenAI-hosted 기본 실행 · 별도 로컬 Docker 복원은 선택 |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · 읽기용 · 실습은 로컬 Python·선택 Docker


## 학습 안내

연결이 끊겨 결과를 보지 못했을 때 입력을 다시 보내면 작업이 중복될 수 있습니다. 이 장은 입력을 한 번 제출하고 관찰 스트림을 닫은 뒤, 같은 Session·Turn에 재연결해 저장된 결과를 확인합니다. 서버 장애나 커널 재시작을 발생시키지 않습니다.

| 실습 | 중단 대상 | 확인할 복구 결과 |
| --- | --- | --- |
| 기본 Agents API | 노트북의 관찰 스트림 | 원래 Turn의 완료 상태·저장 응답·보고서 |
| 선택 Docker | 별도 로컬 컨테이너의 프로세스 | 같은 컨테이너의 파일 유지·새 컨테이너로 결과 복원 |

```bash
uv sync --locked
# Run the hosted API observer-reconnection exercise; Docker is not required.
uv run --group dev python scripts/run_deployment.py --chapter 14 --live
# Run the separate local Docker interruption/restore exercise without API calls.
uv run --group dev python scripts/run_deployment.py --chapter 14 --docker --build
```

기본 RUN_LIVE=False·RUN_DOCKER=False입니다. 기본 API는 OpenAI-hosted 환경이며 Self-hosted executor 키와 Docker가 필요하지 않습니다. Docker 확장은 API와 연결하지 않는 별도 로컬 실습입니다. 실제 청구 대사와 OpenAI 서버 장애 복구는 포함하지 않으며 API 키는 출력하지 않습니다.

[실습 안내와 검증 범위](../../docs/validation.md)

### 이번 장에서 구분할 동작

| 동작 | 의미 |
| --- | --- |
| 관찰 스트림 닫기 | 응답 관찰만 중단하며 취소 요청을 보내지 않음 |
| 같은 Session에 재연결 | 원래 Turn 상태와 저장 결과를 확인 |
| Turn 취소·Session 삭제 | 마지막 자원 정리에서 실행할 수 있는 별도 동작 |
| Docker 재시작·파일 복원 | API와 연결되지 않은 선택 로컬 실습 |


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 실행 모드

로컬 프로젝트에서 `uv sync --locked` 후 `.venv` 커널로 위에서 아래로 실행합니다.

- `RUN_LIVE=True`: Docker 없이 기본 Agents API 재연결 실습을 실행합니다.
- `RUN_DOCKER=True`: 본문 뒤의 별도 로컬 프로세스 중단·파일 복원 예제를 선택합니다.
- 둘 다 True: 기본 API 실습·정리 후 Docker 실습을 실행합니다.
- 둘 다 False: API 준비 셀에서 중단합니다. 오프라인 검증기는 requires_api 셀을 건너뜁니다.

Docker만 실습하려면 라이브러리·설정 셀을 실행하고 API 준비/6단계 셀을 건너뛴 뒤 선택 Docker 절로 이동하세요. CLI의 --docker --build는 이 순서를 자동으로 처리합니다. Colab 독립 실행본은 읽기용이며 이 장의 실행 검증은 로컬 환경을 기준으로 합니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


#### 라이브러리 준비

설정·Docker 보충 예제용 라이브러리를 import합니다. 이 셀은 API 연결이나 컨테이너 시작을 하지 않습니다.


In [ ]:
import json
from tutorial_helpers.common import load_settings
from tutorial_helpers.deployment import DockerLab, build_images, docker, SKILL_COMMAND, JobStore


#### 실행 모드와 입력 설정

기본 hosted API와 별도 Docker 선택을 지정하고 공통 CSV를 읽습니다. API 준비 셀에서 실제 실행을 허용하기 전까지 요청은 전송하지 않습니다.


In [ ]:
RUN_DOCKER = False
RUN_LIVE = False
MODEL_OVERRIDE = ""
CHAPTER = "14_operations"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
root = settings.root
offline_check = os.environ.get("TUTORIAL_OFFLINE_CHECK") == "1"
docker_enabled = RUN_DOCKER and (not offline_check)
csv_bytes = (root / "data/sales.csv").read_bytes()


## 1. API 요청 준비

API 요청·스트리밍·파일 다운로드·자원 정리는 직접 실행합니다. 정답 비교와 검증 기록은 별도 검증 도구에서 처리합니다.


In [ ]:
from tutorial_helpers.common import (
    new_session_metadata, cleanup_resources, cleanup_on_error, check_event, safe_error,
)
from uuid import uuid4
import base64
import time

from IPython.display import Markdown, display


In [ ]:
BASE_INSTRUCTIONS = (
    "You are a careful sales analyst. Use Python "
    "for all calculations. Use only the provided "
    "CSV; never invent rows. Do not browse the web"
    " or delegate. Keep progress commentary short."
    " Return only the JSON object requested by the"
    " user as the final answer, with no Markdown "
    "fences."
)
METRICS_CONTRACT = (
    'Return exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": '
    'integer, "by_product": {product_name: '
    "integer_revenue}}. Revenue is quantity "
    "multiplied by unit_price. Do not round "
    "integer amounts."
)


## 2. 관찰 연결을 끊고 같은 작업의 결과 확인하기

입력 제출 → 관찰 연결 종료 → 같은 Session에 재구독 → 저장된 원래 Turn 확인 → 보고서 다운로드 → 정리 순서입니다. 관찰 스트림을 닫는 단계는 취소 이벤트를 보내지 않습니다. 마지막 정리 단계는 실행 중인 작업이 있으면 취소한 뒤 세션을 삭제할 수 있으므로 서로 구분하세요.

| 단계 | 사용자가 할 일 | 예상 확인 결과 |
| --- | --- | --- |
| 1. 세션 생성 | 식별자와 로컬 상태 파일 확인 | 입력은 아직 제출하지 않음 |
| 2. 한 번 제출·관찰 중단 | 셀 완료까지 기다림; 중단 버튼을 누를 필요 없음 | 입력 수락 여부·원래 Turn ID·관찰 종료 |
| 3. 같은 작업에 재연결 | 다음 셀 실행 | 이미 완료됐으면 대기 생략, 실행 중이면 계속 관찰 |
| 4. 저장된 응답·파일 확인 | 출력 경로의 보고서 열기 | 같은 Turn의 최종 JSON·보고서 다운로드 |
| 5. 운영 정보·usage | ID·Turn 개수·usage 읽기 | root Turn 하나, None은 미보고 |
| 6. 정리·요약 | 마지막에 반드시 실행 | 세션 삭제와 부재 확인 |

셀 사이에서 작업이 완료되는 것은 정상입니다. 이벤트를 다시 재생한다고 가정하지 않으며, 재구독한 상태에서 저장 상태를 조회합니다. 입력을 다시 보내거나 새 세션을 만들지 않습니다. SDK list 응답을 반복해 모든 페이지를 읽고 원래 Turn ID로 결과를 선택합니다.

정상 완료한 셀 사이에는 세션이 남습니다. 중간에 학습을 멈춰도 6번 정리 셀을 실행하세요. 각 단계는 오류·일반적인 중단 시 정리를 시도합니다. 오류 뒤에는 다음 단계로 진행하지 말고 정리 결과를 확인한 뒤 준비 셀부터 새 실행을 시작하세요. 강제 커널 종료의 정리는 보장하지 않습니다. 생성·입력 셀을 반복 실행해 복구를 시도하지 마세요.

관찰 대기 시간은 각 단계에서 새로 측정하므로 셀 사이의 학습 시간은 포함하지 않습니다. 이벤트 검사 기반 600초 예산과 스트림 요청 timeout=90을 사용하며, 엄격한 전체 실행 시간 제한은 아닙니다. 관찰 timeout은 원격 Turn 실패를 뜻하지 않습니다. 이 튜토리얼은 오류 시 자원을 정리하는 정책이며, 운영에서 작업을 보존해 조사하는 정책과 구분합니다.


In [ ]:
environment = {
    "type": "openai_hosted",
    "container_size": "small",
    "network": {"access": "disabled"},
    "files": [
        {
            "type": "inline",
            "path": "/workspace/sales.csv",
            "data": base64.b64encode(csv_bytes).decode(),
        }
    ],
}
task = (
    "Use Python's standard csv module to read /workspace/sales.csv and compute the summary. "
    "Write /workspace/outputs/summary.json containing exactly the requested summary. "
    "Also write /workspace/outputs/report.md in Korean, starting with '# 매출 분석 보고서'. "
    "Include row count, total quantity, total revenue as digits, and every product's revenue. "
    "Read both files back and verify they exist. "
    + METRICS_CONTRACT
)


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


### 이번 실행 상태 준비

새 client로 한 번만 초기화합니다. 진행 중 다시 실행하면 기존 Session과 입력 키를 잃습니다. 1단계는 상태를 초기화하지 않고 중복 Session 생성을 차단합니다. 오류나 정리 후에는 준비 단계부터 새 실행을 시작하세요.


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/deployment-api" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
session = turn = turn_id = usage = None
input_submission_attempts = 0
input_accepted = observer_reconnected = report_downloaded = False
root_turn_count, recovery_source = None, "not_checked"
input_idempotency_key = uuid4().hex
request_state = {"idempotency_key": input_idempotency_key, "submission_state": "not_submitted"}
state_path = output_dir / "request-state.json"

recovered_turn_status = None
downloaded_artifacts = []


### 1. 원래 Session 생성하기

준비한 상태를 유지한 채 입력 없는 Session을 생성합니다. 출력된 Session ID를 다음 단계와 비교하세요. 입력은 2단계에서 한 번만 보냅니다. request-state.json은 Session ID·입력 키·관찰한 Turn ID를 담은 로컬 조사 기록이며 API 키나 프롬프트는 저장하지 않습니다. 생성 셀을 다시 실행하면 중복 생성을 차단하고 오류 시 정리 정책을 적용하므로 기존 셀 실행을 이어가지 마세요.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if session is not None:
        raise RuntimeError("Session creation was already attempted successfully; do not create another session")
    print("[1/6] Creating the original session without input...", flush=True)
    session = client.beta.agents.sessions.create(
        agent={"model": settings.model, "instructions": BASE_INSTRUCTIONS},
        environment=environment,
        metadata=new_session_metadata(CHAPTER, run_id, attempts=attempts, output_dir=output_dir),
    )
    sessions[session.id] = {"deleted": False, "environment_id": session.environment.id}
    request_state.update(session_id=session.id, turn_id=None)
    state_path.write_text(json.dumps(request_state))
    print("Original session ID:", session.id)
    print("Input submitted:", input_accepted)
    print("Local investigation record:", state_path)


### 2. 입력을 한 번 보내고 관찰 중단하기

스트림을 먼저 열고 입력을 전송합니다. root Turn의 실행 중 또는 완료 이벤트를 관찰한 뒤 코드가 스트림을 닫으므로 중단 버튼을 누르지 마세요. 입력 수락은 완료 확인이 아닙니다. 예외가 발생하면 수락 여부가 불명확할 수 있으므로 새 키로 재전송하지 않습니다.

input_idempotency_key는 SDK의 idempotency_key 인자로 전달되어 HTTP 요청의 Idempotency-Key 헤더에 사용됩니다. Session ID·Turn ID나 재개 토큰이 아닙니다. not_submitted는 미전송, attempting은 전송 시도 중으로 수락 여부가 불명확할 수 있는 상태, accepted는 요청 수락을 나타냅니다. accepted도 Turn 완료나 파일 생성을 보장하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if session is None:
        raise RuntimeError("Create the original session in step 1 first")
    if input_submission_attempts:
        raise RuntimeError("Input was already attempted; do not resubmit this cell")
    print("[2/6] Submitting input once, then closing observation...", flush=True)
    started = time.monotonic()
    with client.beta.agents.sessions.events.stream(session.id, timeout=90) as events:
        request_state["submission_state"] = "attempting"
        state_path.write_text(json.dumps(request_state))
        input_submission_attempts += 1
        client.beta.agents.sessions.events.create(
            session.id,
            idempotency_key=input_idempotency_key,
            events=[{
                "type": "agent.session.input.message",
                "input": [{"role": "user", "content": [{"type": "input_text", "text": task}]}],
            }],
        )
        input_accepted = True
        request_state["submission_state"] = "accepted"
        state_path.write_text(json.dumps(request_state))
        for event in events:
            check_event(event)
            if (
                event.type in {"agent.session.turn.in_progress", "agent.session.turn.completed"}
                and event.turn.subagent_id is None
            ):
                turn_id = event.turn.id
                break
            if time.monotonic() - started > 600:
                raise TimeoutError("Initial observation budget exceeded; remote outcome is unknown")
    if turn_id is None:
        roots = [t for t in client.beta.agents.sessions.turns.list(
            session.id, order="asc", timeout=15
        ) if t.subagent_id is None]
        if len(roots) != 1:
            raise RuntimeError("Original root turn could not be identified; do not resubmit input")
        turn_id = roots[0].id
    request_state["turn_id"] = turn_id
    state_path.write_text(json.dumps(request_state))
    print("Input accepted:", input_accepted, "Submission attempts:", input_submission_attempts)
    print("Original turn ID:", turn_id)
    print("Observation stream closed. No cancellation request was sent.")


### 3. 같은 Session·Turn에 재연결하기

새 스트림을 연결한 상태에서 Session과 원래 Turn을 조회합니다. 이미 완료됐으면 이벤트를 기다리지 않고 다음 단계에서 저장된 결과를 읽습니다. 실행 중이면 원래 Turn의 종료를 기다립니다. 이 예제에는 Function Tool이 없으며 예상치 못한 외부 작업을 자동 실행하지 않습니다.

### 재연결 후 상태별 행동

| 조회 결과 | 코드의 행동·사용자가 확인할 내용 |
| --- | --- |
| 원래 Turn 실행 중 | 같은 Turn 관찰을 계속하고 종료 후 저장 상태 재조회 |
| 원래 Turn 완료 | 이벤트 재생을 기다리지 않고 저장 결과 확인 |
| Turn 실패·취소 | 성공으로 처리하지 않고 중단·정리 후 기존 영향 조사 |
| Session 실패 | 반복 재연결을 중단하고 Session 복구 절차 확인 |
| 관찰 종료 후에도 Turn 실행 중 | 완료를 단정하지 않고 오류·정리 정책 적용, 입력 재전송 금지 |
| 입력 수락 불명확 | 3단계로 넘어가지 않고 보존된 ID·서버 상태 조사 |

먼저 스트림을 구독하고 상태를 조회해 둘 사이의 관찰 공백을 줄입니다. observer_reconnected는 연결·조회 성공이며 작업 완료가 아닙니다. 스트림 종료 후에도 원래 Turn을 다시 조회합니다. 이 학습 예제는 오류 시 정리하므로 실행 중인 작업을 보존하는 운영 시스템과 정책이 다릅니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if not input_accepted or turn_id is None:
        raise RuntimeError("Identify the accepted input and original turn in step 2 first")
    observer_reconnected = False
    print("[3/6] Reconnecting to the original session without new input...", flush=True)
    started = time.monotonic()
    terminal_statuses = {"completed", "failed", "cancelled"}
    with client.beta.agents.sessions.events.stream(session.id, timeout=90) as events:
        state = client.beta.agents.sessions.retrieve(session.id, timeout=15)
        if state.status == "failed":
            raise RuntimeError("Session failed; stop reconnecting and investigate session recovery")
        saved = client.beta.agents.sessions.turns.retrieve(turn_id, session_id=session.id, timeout=15)
        if saved.id != turn_id or saved.subagent_id is not None:
            raise RuntimeError("Retrieved turn is not the original root turn")
        observer_reconnected = True
        print("Original turn status on reconnect:", saved.status)
        if any(action.type != "environment_connection" for action in state.required_actions):
            raise RuntimeError("Unexpected external action; reconcile its outcome without replaying input")
        if saved.status in terminal_statuses:
            recovery_source = "persisted_state"
            print("Turn is already terminal; reading persisted results without waiting for replay.")
        else:
            recovery_source = "live_events_then_persisted_state"
            print("Original turn is still active; continuing observation...", flush=True)
            for event in events:
                check_event(event)
                if event.type == "agent.session.turn.output_text.delta":
                    print(event.delta, end="", flush=True)
                elif event.type in {
                    "agent.session.turn.completed", "agent.session.turn.failed",
                    "agent.session.turn.cancelled",
                }:
                    if event.turn.id == turn_id and event.turn.subagent_id is None:
                        break
                elif event.type == "agent.session.requires_action":
                    raise RuntimeError("External work needs a responder; input is not retried")
                if time.monotonic() - started > 600:
                    raise TimeoutError("Recovery observation budget exceeded; remote outcome is unknown")
    # A closed observer does not establish the final outcome of the work.
    recovered = client.beta.agents.sessions.turns.retrieve(turn_id, session_id=session.id, timeout=15)
    if recovered.id != turn_id or recovered.subagent_id is not None:
        raise RuntimeError("Retrieved turn is not the original root turn")
    recovered_turn_status = recovered.status
    if recovered.status != "completed":
        raise RuntimeError("Original turn is " + recovered.status + "; do not resubmit input")
    print("\nReconnected session ID:", session.id)
    print("Persisted turn status:", recovered_turn_status)
    print("Original turn ID:", turn_id, "Result source:", recovery_source)


### 4. 저장된 응답과 보고서 확인하기

실시간 출력이 없었어도 저장된 원래 Turn과 최종 응답을 조회합니다. SDK 페이지 반복으로 전체 이력을 읽고 root Turn이 하나인지 확인합니다. 해당 Turn의 파일만 다운로드한 뒤 출력된 보고서 경로를 여세요. 모델의 완료 문구와 파일 다운로드는 다른 확인 단계입니다.

phase="final_answer"는 중간 설명과 최종 응답을 구분합니다. 원래 Turn에 속한 assistant 최종 메시지만 SDK 순서대로 조합하며, 선택한 Item ID도 표시합니다. JSON 계약에 맞지 않으면 성공으로 처리하지 않습니다. Artifact도 같은 Turn과 정확한 경로로 선택합니다. 파일마다 ID·Turn·경로를 출력하고 두 파일 다운로드가 끝난 뒤에만 report_downloaded를 True로 표시합니다. 아래에 요약 파일 내용과 보고서 본문이 표시되므로 출력 경로를 따로 열지 않아도 확인할 수 있습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    report_downloaded = False
    downloaded_artifacts = []
    if not observer_reconnected or turn_id is None:
        raise RuntimeError("Reconnect to the original turn in step 3 first")
    print("[4/6] Retrieving persisted results from the original turn...", flush=True)
    turn = client.beta.agents.sessions.turns.retrieve(turn_id, session_id=session.id, timeout=15)
    if turn.id != turn_id or turn.subagent_id is not None:
        raise RuntimeError("Retrieved turn is not the original root turn")
    if turn.status != "completed":
        raise RuntimeError("Original root turn did not complete: " + turn.status)
    saved_turns = list(client.beta.agents.sessions.turns.list(session.id, order="asc", timeout=15))
    roots = [t for t in saved_turns if t.subagent_id is None]
    root_turn_count = len(roots)
    if [t.id for t in roots] != [turn_id]:
        raise RuntimeError("Expected exactly one original root turn")
    items = list(client.beta.agents.sessions.items.list(session.id, order="asc", limit=100, timeout=15))
    messages = [item for item in items if item.turn_id == turn_id and item.type == "message"
                and item.role == "assistant" and item.phase == "final_answer"]
    if not messages:
        raise RuntimeError("Completed turn has no persisted final answer")
    print("Selected final-answer item IDs:", [message.id for message in messages])
    try:
        actual = json.loads("".join(message.output_text for message in messages))
    except json.JSONDecodeError as error:
        raise RuntimeError("Persisted final answer is not the requested JSON; inspect selected items") from error
    artifacts = list(client.beta.agents.sessions.artifacts.list(session.id, order="asc", timeout=15))
    downloads = output_dir / "downloads"
    downloads.mkdir(exist_ok=True)
    for name in ("summary.json", "report.md"):
        matches = [a for a in artifacts if a.turn_id == turn_id and a.path == "/workspace/outputs/" + name]
        if len(matches) != 1:
            raise RuntimeError("Expected one artifact for the original turn and path")
        artifact = matches[0]
        with client.beta.agents.sessions.artifacts.with_streaming_response.content(
            artifact.id, session_id=session.id, timeout=30,
        ) as content:
            content.stream_to_file(downloads / name)
        downloaded_artifacts.append({"id": artifact.id, "turn_id": artifact.turn_id, "path": artifact.path})
        print("Downloaded artifact:", downloaded_artifacts[-1])
    report_downloaded = True
    print("Recovered turn status:", turn.status)
    print("Root turn count:", root_turn_count)
    print("Summary:", actual)
    print("Downloaded summary:", downloads / "summary.json")
    print("Downloaded report:", downloads / "report.md")

    print("Downloaded summary contents:", json.loads((downloads / "summary.json").read_text()))
    display(Markdown((downloads / "report.md").read_text()))


### 5. 운영 정보와 usage 해석하기

입력 제출 시도 한 번·같은 Session/Turn·root Turn 하나를 함께 확인합니다. turn.usage를 그대로 표시하며 None은 미보고, 숫자 0은 보고된 0입니다. 사용량 필드의 존재와 실제 청구 대사는 구분합니다. 이 단계는 추가 입력을 보내지 않습니다.

요청 수락 → 관찰 재연결 → 저장된 Turn 완료 → 두 파일 다운로드는 서로 다른 증거입니다. 로컬 기록은 조사에 도움을 주지만 서버 조회 결과를 대신하지 않습니다. 다음 정리 단계에서는 취소·삭제가 발생할 수 있으므로 그 전에 실행 결과를 확인하세요.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    print("[5/6] Inspecting operational evidence and reported usage...")
    if turn is None or not report_downloaded:
        raise RuntimeError("Retrieve the completed turn and both files in step 4 first")
    usage = turn.usage.to_dict() if turn.usage is not None else None
    print("Original session ID:", session.id)
    print("Original turn ID:", turn_id)
    print("Input submission attempts:", input_submission_attempts)
    print("Input accepted:", input_accepted)
    print("Observer reconnected:", observer_reconnected)
    print("Root turn count:", root_turn_count)
    print("Usage (None means not reported):", usage)
    print("Reported usage does not verify an invoice or a cost cap.")

    print("Persisted turn status:", recovered_turn_status)
    print("Both report files downloaded:", report_downloaded)
    print("Local request record (not server state):", request_state)


### 6. 자원 정리와 최종 요약

학습을 중간에 멈춰도 이 셀은 실행하세요. 실행 중이면 정리 helper가 취소를 시도한 뒤 세션 삭제·부재를 확인합니다. 관찰 연결만 닫은 2단계와 다릅니다. 원래 오류가 있으면 정리 오류로 덮지 않고 추가 정보로 남깁니다. 로컬 조사 기록은 삭제된 세션을 자동 재개하는 체크포인트가 아닙니다.


In [ ]:
try:
    print("[6/6] Cleaning up the original session...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
request_state["cleanup"] = "verified" if sessions else "not_needed"
state_path.write_text(json.dumps(request_state))
print({
    "turn_status": turn.status if turn is not None else "not_completed",
    "input_submission_attempts": input_submission_attempts,
    "observer_reconnected": observer_reconnected,
    "root_turn_count": root_turn_count,
    "report_downloaded": report_downloaded,
    "session_cleanup": request_state["cleanup"],
})


## 3. 운영 확장: 중복 방지와 복구 기록

이 장의 기본 경로는 같은 Python 커널에서 관찰자만 재연결합니다. request-state.json에는 입력 전송 전에 키를 기록하고 수락 뒤 상태·관찰한 Turn ID를 갱신합니다. attempting 상태에서 중단됐다면 수락 여부를 단정할 수 없습니다. 새 키로 재전송하지 말고 보존된 ID와 서버 상태를 조사해야 합니다. 이 실습은 오류 시 세션을 정리하므로 저장된 ID만으로 자동 재개할 수 없습니다.

idempotency_key는 중복 요청 방지용 식별자이며 Turn 완료를 뜻하지 않습니다. 요청 수락 → 원래 Turn 식별 → 저장된 종료 상태 → 파일 다운로드를 각각 확인하세요. 프로덕션의 커널/프로세스 재시작 복구에는 지속 작업 저장소·소유권·오류 시 보존/정리 정책이 추가로 필요합니다.

아래 JobStore는 로컬 SQLite에서 같은 작업의 생성 권한을 한 번만 부여하는 보충 예제입니다. Session 생성·입력 전송은 하지 않습니다. 새 실행은 새 DB를 만들므로 실행 전체에 걸친 중복 방지나 분산 exactly-once를 보장하지 않습니다.

동일 DB 경로를 다시 열어 같은 작업·소유자로 claim해야 기존 소유권을 확인할 수 있습니다. 아래 출력의 DB 경로·소유자·작업 상태를 비교하세요. 이 로컬 잠금과 API 요청의 중복 방지 키는 범위가 다르며, JobStore가 원격 Turn을 자동 재개하지 않습니다.


In [ ]:
ownership_run_id = uuid4().hex
ownership_path = root / "outputs/deployment-local/14_operations" / ownership_run_id / "jobs.sqlite3"
ownership_store = JobStore(ownership_path)
first_claim = ownership_store.claim("sales-2026-01", ownership_run_id)
repeated_claim = JobStore(ownership_path).claim("sales-2026-01", ownership_run_id)
print("First claim grants creation:", first_claim["created"])
print("Repeated claim grants creation:", repeated_claim["created"])
print("No API request was sent by this ownership example.")

print("Ownership DB:", ownership_path)
print("Owner:", ownership_run_id, "Stored state:", repeated_claim["state"])


### 추가 도구가 있는 작업을 복구한다면

기본 예제는 Function Tool을 등록하지 않으므로 pending 도구 결과 복구 코드를 메인 경로에 넣지 않습니다. 도구를 추가하는 경우에는 Session의 required_actions를 조회하고 session/turn/call 식별자와 원래 인자로 저장된 결과를 찾아 반환해야 합니다. 결과를 모른다는 이유로 부작용이 있는 도구를 다시 실행하면 안 됩니다.

[7장 Function Tool](07_function_tools.ipynb)은 SDK의 도구 결과 자동 반환을, [11장 승인](11_guardrails_approvals.ipynb)의 ResultCache는 저장한 호출 결과 재사용을 보여줍니다. 도구 실행 성공 후 결과 저장 전에 중단된 경우는 캐시만으로 해결하지 못하므로 업무 시스템과 대조해야 합니다. 이 장은 이 조합을 실제 실행하는 예제가 아닙니다.


## 4. 선택 확장: 별도 Docker 프로세스 중단·파일 복원

기본 API 실습과 세션 정리를 끝낸 뒤 선택적으로 실행합니다. 이 Docker worker는 앞선 OpenAI-hosted 세션의 실행 환경이 아닙니다. 로컬에서 프로세스를 실제로 종료하고 같은 컨테이너 재시작과 새 컨테이너 복원을 비교합니다. 앞서 만든 보고서의 복원이며 중단된 분석의 자동 재실행을 구현하지 않습니다.

출력 순서: 작업 소유권 확보·보고서 저장 → 대기 프로세스 시작·컨테이너 종료 → 같은 컨테이너의 started 파일 존재와 finished 파일 부재 → 새 컨테이너에 저장 보고서 복원 → Docker 정리. 리소스 정리를 보장하기 위해 하나의 DockerLab context 안에서 실행합니다. API 변수와 구분한 docker_store를 사용합니다.

| 확인 지점 | 예상 결과 | 의미 |
| --- | --- | --- |
| 중단 전 | 보고서 저장 완료 | 복원할 대상은 이미 생성된 파일 |
| 같은 컨테이너 재시작 | started 존재·finished 부재 | 파일 보존이 중단 프로세스 재개를 뜻하지 않음 |
| 새 컨테이너 | 저장한 summary.json 복원 | 분석 재실행이 아닌 이전 결과 복원 |
| JobStore completed → cleaned | 복원 단계 완료 후 자원 정리 | 중단한 계산이 재개·완료됐다는 의미가 아님 |


In [ ]:
if docker_enabled:
    build_images(root, executor=False)
    with DockerLab("14_operations", root) as lab:
        print("[Docker 1/4] Claiming local work and saving the report...", flush=True)
        _, worker = lab.package_environment()
        docker_store = JobStore(lab.output_dir / "jobs.sqlite3")
        assert docker_store.claim("sales-2026-01", lab.run_id)["created"]
        assert not docker_store.claim("sales-2026-01", lab.run_id)["created"]
        docker_store.transition("sales-2026-01", lab.run_id, "provisioning", "running", worker)
        lab.install(worker)
        lab.execute(worker, SKILL_COMMAND)
        lab.collect_report(worker)

        print("[Docker 2/4] Starting a waiting process and terminating its container...", flush=True)
        wait_script = (
            "from pathlib import Path; import time; "
            "Path('/workspace/started').write_text('started'); time.sleep(300); "
            "Path('/workspace/finished').write_text('finished')"
        )
        docker(["exec", "-d", worker, "python", "-c", wait_script])
        started_check = "from pathlib import Path; assert Path('/workspace/started').exists()"
        for _ in range(30):
            if lab.execute(worker, ["python", "-c", started_check], check=False).returncode == 0:
                break
            time.sleep(0.1)
        else:
            raise TimeoutError("fault_injection_not_started")
        docker(["kill", worker])
        docker_store.transition("sales-2026-01", lab.run_id, "running", "needs_reconciliation")

        print("[Docker 3/4] Restarting the same container and inspecting retained files...", flush=True)
        lab.start(worker)
        unfinished_check = (
            started_check + "; assert not Path('/workspace/finished').exists()"
        )
        lab.execute(worker, ["python", "-c", unfinished_check])
        print("Retained started marker; no finished marker was produced.")

        print("[Docker 4/4] Restoring the saved report into a replacement...", flush=True)
        replacement = lab.create("replacement")
        lab.start(replacement)
        empty_check = "from pathlib import Path; assert not Path('/workspace/outputs/summary.json').exists()"
        lab.execute(replacement, ["python", "-c", empty_check])
        docker(["cp", lab.output_dir / "downloaded", f"{replacement}:/workspace/restored"])
        restored = lab.output_dir / "restored-summary.json"
        lab.copy_out(replacement, "/workspace/restored/summary.json", restored)
        docker_store.transition("sales-2026-01", lab.run_id, "needs_reconciliation", "completed")
        print("Restored summary:", restored)
        print("Restoration completed; the interrupted process was not resumed.")
    docker_store.transition("sales-2026-01", lab.run_id, "completed", "cleaned")
    print("Docker resources cleaned up; local ownership state: cleaned")
else:
    print("Optional Docker example skipped. Set RUN_DOCKER=True to run it separately.")


## 완료 기준과 다음 단계

기본 실습은 입력 제출 시도 한 번, 같은 Session·원래 root Turn 하나, 재연결 후 저장된 응답과 보고서 다운로드, usage 표시, 세션 삭제·부재 확인으로 완료합니다. 모델의 완료 문구만으로 성공을 판단하지 않습니다. Docker 확장의 파일 유지·복원 결과는 별도로 확인합니다.

실행 사본 검증기는 세션 삭제 전에 원격 이력·명령·보고서를 검사하고 receipt를 저장합니다. 노트북 수동 실행은 단계별 출력으로 확인하며 정답 비교·receipt 생성은 하지 않습니다. 오류 시 정리를 시도하지만 강제 커널 종료나 프로세스 손실의 정리는 보장하지 않습니다.

[3장 세션·실행 상태](03_sessions.ipynb)에서 Event/Item·취소/삭제를 복습하고, [12장 Self-hosted](12_self_hosted.ipynb)의 executor 재연결과 [13장 관측](13_hooks_telemetry.ipynb)의 별도 애플리케이션 span을 이 장의 관찰자 재연결과 구분하세요.

- [실습 안내와 검증 범위](../../docs/validation.md)

다음 [15장 Multi-agent](15_multi_agent.ipynb)에서는 원래 Turn의 실행·파일을 구분한 경험을 여러 담당의 실행과 통합 결과 확인으로 확장합니다. 15장은 새 OpenAI-hosted Session을 사용하며 Docker가 필요하지 않습니다.


## 공식 참고 자료

### Agents API 전용 문서

- [Events and items](https://developers.openai.com/api/docs/guides/agents-api/sessions/events) — 이벤트 스트림·완료 상태·저장된 items와 turns 조회.
- [Errors and recovery](https://developers.openai.com/api/docs/guides/agents-api/errors) — 오류 분류·저장 상태 확인·안전한 재시도.
- [Sandbox lifecycle](https://developers.openai.com/api/docs/guides/agents-api/environments/lifecycle) — compute 시작·재연결·종료와 API 세션 정리의 구분.
- [Observability and usage](https://developers.openai.com/api/docs/guides/agents-api/observability) — 실행 기록·Dashboard·Turn usage와 비용 해석.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
